# 2. Eksplorasi dan Pemahaman Data
Bagian A melihat ukuran, harga, sebaran, tren, dan inflasi dari `data_bersih.csv`. Bagian B memetakan seluruh kolom data mentah, lalu menelaah kualitas, kepercayaan, dan volatilitas yang tidak terlihat dari harga saja.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

raw_path = '../data/raw/IDN_RTFP_mkt_2007_2026-08-24.csv'
data_path = '../data/processed/data_bersih.csv'

KOL_PASAR = 'mkt_name'
KOL_PROVINSI = 'adm1_name'
KOL_TANGGAL = 'date'
KOMODITAS = ['eggs', 'garlic', 'meat_beef', 'meat_chicken', 'meat_chicken_broiler', 'oil', 'onions', 'rice', 'sugar']
ITEM = KOMODITAS + ['food_price_index']

## Bagian A. Eksplorasi data

In [ ]:
df = pd.read_csv(data_path, parse_dates=[KOL_TANGGAL])
harga = [f'c_{k}' for k in KOMODITAS if f'c_{k}' in df.columns]
inflasi = [c for c in df.columns if c.startswith('inflation_')]
print(f'{len(df):,} baris | {df[KOL_PASAR].nunique()} pasar | {df[KOL_PROVINSI].nunique()} provinsi')
print(f'Periode: {df[KOL_TANGGAL].min():%b %Y} - {df[KOL_TANGGAL].max():%b %Y}')
print(f'Komoditas: {len(harga)} | kolom inflasi: {len(inflasi)}')
df.head()

## Statistik deskriptif harga penutupan (c_)

In [ ]:
df[harga].describe().T.round(0)

## Missing value pada harga dan inflasi

In [ ]:
kosong = df[harga + inflasi].isna().mean().mul(100).round(1).sort_values(ascending=False)
kosong.to_frame('persen_kosong')

In [ ]:
tahun_inflasi = df.groupby(df[KOL_TANGGAL].dt.year)[inflasi[0]].apply(lambda s: s.notna().mean() * 100).round(1) if inflasi else None
tahun_inflasi

## Jumlah pasar yang tercatat per tahun

In [ ]:
pasar_per_tahun = df.groupby(df[KOL_TANGGAL].dt.year)[KOL_PASAR].nunique()
pasar_per_tahun.plot(kind='bar', figsize=(10, 4), title='Jumlah pasar per tahun')
plt.ylabel('Jumlah pasar')
plt.show()

## Tren rata-rata harga bulanan

In [ ]:
df['bulan'] = df[KOL_TANGGAL].dt.to_period('M').dt.to_timestamp()
tren = df.groupby('bulan')[harga].mean()
fig, axes = plt.subplots(3, 3, figsize=(15, 10), sharex=True)
for ax, kol in zip(axes.ravel(), harga):
    tren[kol].plot(ax=ax, title=kol.replace('c_', ''))
    ax.set_xlabel('')
plt.tight_layout()
plt.show()

## Sebaran harga dan outlier (hanya dilihat, tidak diubah)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
for ax, kol in zip(axes.ravel(), harga):
    ax.boxplot(df[kol].dropna(), vert=False)
    ax.set_title(kol.replace('c_', ''))
plt.tight_layout()
plt.show()

In [ ]:
def jumlah_outlier(s):
    s = s.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    batas = 1.5 * (q3 - q1)
    return int(((s < q1 - batas) | (s > q3 + batas)).sum())

pd.Series({k: jumlah_outlier(df[k]) for k in harga}, name='jumlah_outlier_iqr').to_frame()

## Harga rata-rata beras per provinsi

In [ ]:
if 'c_rice' in df.columns:
    rata_prov = df.groupby(KOL_PROVINSI)['c_rice'].mean().sort_values()
    rata_prov.tail(15).plot(kind='barh', figsize=(8, 6), title='15 provinsi dengan harga beras tertinggi (IDR/kg)')
    plt.show()

## Inflasi tahunan

In [ ]:
for kol in inflasi[:1]:
    df.groupby('bulan')[kol].mean().plot(figsize=(12, 4), title=f'Rata-rata {kol}')
    plt.axhline(5, color='orange', linestyle='--')
    plt.axhline(10, color='red', linestyle='--')
    plt.ylabel('Inflasi (%)')
    plt.show()

## Bagian B. Pemahaman data (88 fitur)

### Kamus variabel: kelompok kolom pada data mentah

In [ ]:
kolom_mentah = pd.read_csv(raw_path, nrows=0).columns.str.strip().tolist()
identitas = ['ISO3', 'country', 'adm1_name', 'adm2_name', 'mkt_name', 'lat', 'lon', 'geo_id']
waktu = ['price_date', 'year', 'month']
metadata = ['currency', 'components', 'start_dense_data', 'last_survey_point', 'data_coverage',
            'data_coverage_recent', 'index_confidence_score', 'spatially_interpolated']
kelompok = {
    'Identitas dan lokasi': identitas,
    'Waktu': waktu,
    'Metadata dan kualitas': metadata,
    'Harga mentah (survei langsung)': [k for k in KOMODITAS if k in kolom_mentah],
    'Harga open (o_)': [c for c in kolom_mentah if c.startswith('o_')],
    'Harga high (h_)': [c for c in kolom_mentah if c.startswith('h_')],
    'Harga low (l_)': [c for c in kolom_mentah if c.startswith('l_')],
    'Harga close (c_)': [c for c in kolom_mentah if c.startswith('c_')],
    'Inflasi tahunan (inflation_)': [c for c in kolom_mentah if c.startswith('inflation_')],
    'Skor kepercayaan (trust_)': [c for c in kolom_mentah if c.startswith('trust_')],
}
ringkas = pd.DataFrame({'jumlah_kolom': {k: len([c for c in v if c in kolom_mentah]) for k, v in kelompok.items()}})
print(f'Total kolom mentah: {len(kolom_mentah)} | terpetakan: {ringkas.jumlah_kolom.sum()}')
ringkas

In [ ]:
tersisa = [c for c in kolom_mentah if c in df.columns or (c == 'price_date' and KOL_TANGGAL in df.columns)]
print(f'Kolom setelah cleaning: {df.shape[1]} (kolom satu nilai dibuang)')
print('Kolom dibuang:', [c for c in kolom_mentah if c not in df.columns and c != 'price_date'])

### Kepercayaan data (trust_) dan skor indeks

In [ ]:
trust = [f'trust_{k}' for k in ITEM if f'trust_{k}' in df.columns]
df[trust].describe().T[['mean', 'min', 'max']].round(2)

In [ ]:
df['tahun'] = df[KOL_TANGGAL].dt.year
tren_trust = df.groupby('tahun')[trust].mean().mean(axis=1)
tren_trust.plot(figsize=(10, 4), marker='o', title='Rata-rata skor kepercayaan per tahun')
plt.ylabel('Skor trust')
plt.grid(True)
plt.show()

In [ ]:
trust_komoditas = df[trust].mean().rename(lambda c: c.replace('trust_', '')).sort_values()
trust_komoditas.plot(kind='barh', figsize=(8, 5), title='Rata-rata skor kepercayaan per komoditas')
plt.show()

In [ ]:
kolom_meta = [c for c in ['index_confidence_score', 'data_coverage', 'data_coverage_recent'] if c in df.columns]
if kolom_meta:
    pasar = df.groupby(KOL_PASAR)[kolom_meta].mean()
    print(pasar.describe().round(2))
else:
    print('Skor indeks dan cakupan data bernilai tunggal sehingga sudah dibuang saat cleaning')

### Interpolasi spasial

In [ ]:
if 'spatially_interpolated' in df.columns:
    status_pasar = df.groupby(KOL_PASAR)['spatially_interpolated'].max()
    print(f'Pasar dengan harga hasil interpolasi: {int(status_pasar.sum())} dari {len(status_pasar)} ({status_pasar.mean() * 100:.1f}%)')
    gabung = df.groupby('spatially_interpolated')[trust].mean().mean(axis=1)
    print('Rata-rata trust: interpolasi = 0 vs 1')
    print(gabung.round(2).to_string())
else:
    print('Kolom spatially_interpolated bernilai tunggal dan sudah dibuang saat cleaning')

### Harga mentah vs harga estimasi
Harga mentah hanya terisi saat ada survei langsung; harga close (c_) adalah estimasi yang selalu tersedia.

In [ ]:
mentah = [k for k in KOMODITAS if k in df.columns]
terisi = df[mentah].notna().mean().mul(100).round(1).sort_values()
terisi.plot(kind='barh', figsize=(8, 5), title='Persen baris dengan observasi harga langsung')
plt.xlabel('Persen terisi')
plt.show()

In [ ]:
df[mentah].notna().groupby(df['tahun']).mean().mean(axis=1).mul(100).round(1).to_frame('persen_observasi_langsung')

### Volatilitas dalam bulan
Rentang harga = (high - low) / close; makin besar, makin tidak stabil harga dalam satu bulan.

In [ ]:
rentang = {}
for k in ITEM:
    h, l, c = f'h_{k}', f'l_{k}', f'c_{k}'
    if {h, l, c}.issubset(df.columns):
        rentang[k] = ((df[h] - df[l]) / df[c] * 100).mean()
rentang = pd.Series(rentang).sort_values()
rentang.plot(kind='barh', figsize=(8, 5), title='Rata-rata rentang harga dalam bulan (%)')
plt.show()

In [ ]:
df['rentang_indeks'] = (df['h_food_price_index'] - df['l_food_price_index']) / df['c_food_price_index'] * 100
df.groupby(KOL_PROVINSI)['rentang_indeks'].mean().sort_values(ascending=False).head(10).round(2).to_frame('rentang_rata_rata_persen')

### Hubungan kepercayaan dan volatilitas

In [ ]:
pasar_ringkas = df.groupby(KOL_PASAR).agg(trust=('trust_food_price_index', 'mean'), rentang=('rentang_indeks', 'mean'))
print('Korelasi trust dan rentang harga antar pasar:', round(pasar_ringkas.corr().iloc[0, 1], 3))